In [ ]:
import numpy as np
np.random.default_rng(2159)

PYTHONHASHSEED=0

In [ ]:
%matplotlib inline

#load packages
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os

sc.settings.verbosity = 3   
sc.set_figure_params(scanpy=True, fontsize=12, dpi_save=180, format='svg', transparent=True)
sc.settings.set_figure_params(dpi=80)

In [ ]:
my_c = ['#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4']

# Load raw counts

In [ ]:
adata_filt = sc.read_h5ad('Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata_raw = sc.read_h5ad('Postnatal_unfiltered.h5ad')

In [ ]:
adata_annotated = sc.read_h5ad('Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_raw.var_names_make_unique()

adata_annotated.var_names_make_unique()

In [ ]:
adata_all = adata_raw[adata_filt.obs.index]

In [ ]:
adata_all.obs = adata_filt.obs

In [ ]:
adata_all

# Add metadata

In [ ]:
adata_all.obs['Cell_type'] = adata_annotated.obs['Cell_type']

# Subset data only for Immune cells

In [ ]:
adata_all

In [ ]:
adata = adata_all[adata_all.obs['Cell_type'].isin(['Immune cells'])]

In [ ]:
adata

In [ ]:
adata.write_h5ad('Immune_Postnatal_unfiltered.h5ad')

In [ ]:
adata = sc.read_h5ad('Immune_Postnatal_unfiltered.h5ad')

In [ ]:
adata

## Ploting data before QC

In [ ]:
sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'],
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_rawdata.svg')

In [ ]:
adata.obs['Log10_nGenes'] = np.log10(adata.obs['n_genes_by_counts'])

In [ ]:
vars_plot = ['total_counts', 'Log10_nGenes', 'n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb']

In [ ]:
sc.pl.violin(adata, keys=vars_plot, rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_rawdata.svg')

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 5), sharex=True)

groupby_var = "batch"

# Loop over genes and groupby variables to create violin plots
for i, v in enumerate(vars_plot):
    if i != 1:
        ax = axes[i] if 6 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0)  # Adjust the y-limits as needed for each gene
    else:
        ax = axes[i] if 5 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0, top=5)  # Adjust the y-limits as needed for each gene

# Add a common x-axis label and display the plot
plt.tight_layout()  # Adjust subplot layout to prevent overlap
plt.show()

fig.savefig('violinplots_immune_sample_metrics.svg', format='svg', dpi=180)

In [ ]:
sc.pl.violin(adata, ["Xist-counts", "percent_chrY"], jitter=0, groupby = 'batch', rotation= 90, palette=my_c)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'Xist-counts'], vmax=1, palette=my_c)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'percent_chrY'], vmax=0.1)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')

plt.show()
    
fig.savefig('immune_barplots_rawdata.svg', format='svg', dpi=180)

In [ ]:
sc.pl.highest_expr_genes(adata,n_top=20)

In [ ]:
sc.pl.violin(adata, ['S_score', 'G2M_score'], jitter=0, groupby = 'batch', rotation=90, palette=my_c)

In [ ]:
adata

# QC

## Data normalization and logaritmization

In [ ]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata)

**Creating raw data object**

In [ ]:
adata.raw = adata.copy()

## Searching for doublets

In [ ]:
##  Predict doublets
import scrublet as scr

scrub = scr.Scrublet(adata.raw.X)
doublet_score, predicted_doublets = scrub.scrub_doublets()

In [ ]:
%matplotlib inline

In [ ]:
scrub.plot_histogram()

In [ ]:
adata.obs['predicted_doublets'] = predicted_doublets

In [ ]:
sum(adata.obs['predicted_doublets'])

In [ ]:
## add in column with singlet/doublet instead of True/False
adata.obs['doublet_info'] = adata.obs["predicted_doublets"].astype(str)
sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, groupby = 'doublet_info', rotation=45, palette=my_c, 
             save='_immune_doublets.svg')

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['doublet_info'], palette=my_c, save='_immune_doublets.svg')

In [ ]:
adata = adata[adata.obs['doublet_info'] == 'False',:]
print(adata.shape)

In [ ]:
adata

## Saving data without doublets

In [ ]:
adata.write_h5ad('Immune_Postnatal_noDBL.h5ad')

In [ ]:
adata = sc.read_h5ad('Immune_Postnatal_noDBL.h5ad')

In [ ]:
adata

## Filtering cells and genes

In [ ]:
sc.pp.filter_cells(adata,min_genes = 250)
sc.pp.filter_genes(adata,min_cells = 3)

In [ ]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_prefilter.svg')

In [ ]:
adata = adata[adata.obs['total_counts'] < 20000,:] 

In [ ]:
adata = adata[adata.obs['n_genes_by_counts'] < 4000, :]

In [ ]:
adata = adata[adata.obs['pct_counts_Mito'] < 10,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Hb'] < 0.05,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Ribo'] < 20,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Ribo'] > 2.5,:]

In [ ]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_afterfilter.svg')

In [ ]:
print("Remaining cells %d" % adata.n_obs)

In [ ]:
adata

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
    
fig.savefig('immune_barplots_afterfilter.svg', format='svg')

## Saving filtered data

In [ ]:
adata.write_h5ad('Immune_Postnatal_noDBL_filtered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Immune_Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata

In [ ]:
adata

# Dimensionality redutcion

## HVG

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat_v3', n_top_genes=4000)
print("Highly variable genes: %d"%sum(adata.var.highly_variable))

In [ ]:
sc.pl.highly_variable_genes(adata, save='_immune_HVG.svg')

In [ ]:
var_genes_all = adata.var.highly_variable

In [ ]:
adata = adata[:, var_genes_all]

In [ ]:
adata

In [ ]:
sc.pp.scale(adata,max_value=10)

## PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca(adata, color='batch', components = ['1,2','3,4','5,6','7,8'], ncols=2, wspace=0.8, palette=my_c, 
          save='_immune_pca.svg')

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='_immune_n_pcs.svg')

# Clustering

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["batch", "Age"], size=15, wspace=0.8, ncols=2, palette=my_c, save='_immune_precorrection.svg')

## Batch correction

### Harmony

In [ ]:
from harmony import harmonize

In [ ]:
Z = harmonize(adata.obsm['X_pca'], adata.obs, batch_key = 'batch')
adata.obsm['X_harmony'] = Z

In [ ]:
print("Calculating UMAP")
sc.pp.neighbors(adata, use_rep="X_harmony")
sc.tl.umap(adata)

In [ ]:
print("Ploting UMAP")
sc.pl.umap(adata, color=["batch", "Age"], size=15, wspace=0.8, ncols=2, palette=my_c, save='_immune_precorrection.svg')

## Clustering

In [ ]:
# clustering
for res in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    sc.tl.leiden(adata, resolution=res,key_added = 'leiden_%s'%res)

In [ ]:
#plot
sc.pl.umap(adata,color=['leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 
                        'leiden_0.8', 'leiden_1.0'], ncols=4, wspace=1, cmap='my_c')

In [ ]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='leiden_0.8', save='_immune_afterclustering_checking_doublets.svg')

## Saving clsutered data

In [ ]:
adata.write_h5ad('Immune_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Immune_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata

### Leiden_0.1

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.1', method='wilcoxon', key_added = "wilcoxon_01", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_01" , groupby="leiden_0.1", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.2

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.2', method='wilcoxon', key_added = "wilcoxon_02", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_02" , groupby="leiden_0.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.3

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.3', method='wilcoxon', key_added = "wilcoxon_03", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_03" , groupby="leiden_0.3", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.4

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.4', method='wilcoxon', key_added = "wilcoxon_04", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_04" , groupby="leiden_0.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.5

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.5', method='wilcoxon', key_added = "wilcoxon_05", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_05" , groupby="leiden_0.5", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.6

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_06", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_06" , groupby="leiden_0.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.8

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.8', method='wilcoxon', key_added = "wilcoxon_08", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_08" , groupby="leiden_0.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.0

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.0', method='wilcoxon', key_added = "wilcoxon_1", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_1" , groupby="leiden_1.0", cmap="YlGnBu", vmin=0, vmax=10)

## Saving clsutered data

In [ ]:
adata.write_h5ad('Immune_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading clustered data

In [ ]:
adata = sc.read_h5ad('Immune_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata

In [ ]:
adata

In [ ]:
adata_all = sc.read_h5ad('Immune_Postnatal_unfiltered.h5ad')

In [ ]:
markers = {"Epithelial cells": ['Gata3','Atp10b','Cidea', "Krt18","Htr2c","Car12","Clic6", "Folr1","Igfbp2",'Otx2','Chchd10','Clu','Calml4'],
           "Mesenchymal cells": ['Pdgfrb','Mgp','Cfh','Dcn','Tgfbi','Moxd1','Cp','Bgn','Cxcl12','Col3a1'],
           "Endothelial cells": ["Egfl7","Kdr","Pecam1", "Cd34", "Plvap",'Cdh5','Flt1','Igfbp3','Ly6a','Ly6c1'],
           "Immune cells": ["C1qa",'Tyrobp','Fcer1g','Ctss','C1qb','Ptprc','C1qc','Cd83','Lyz2','Laptm5'],
           "NeuroGlia": ['Tubb2b','Gpm6b','Rtn1','Olig1','Fabp7','Bcan','Cnp','Scrg1','Opcml','Pcsk1n'], 
           "NIPC": ["Cdca7", "2810417H13Rik", "Rrm2", "Ube2c", "Top2a", "Birc5", "P2rx3"]}

In [ ]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=6)

In [ ]:
markers1 = {'Ciliogenesis': ['Shisa8', 'Ccdc67', 'Ccno', 'Mcidas'], 
                'Mesenchymal cells': ['Dcn', 'Adam12', 'Cxcl12', 'Lum'], 
                'Epithelial cells': ['Htr2c', 'Enpp2', 'Kcnj13', 'Aqp1', 'Gata3'], 
                'Endothelial cells': ['Plvap', 'Flt1', 'Pecam1'],
                'Neuro / Glial': ['Slc1a3', 'Olig1', 'Olig2', 'Dlx1', 'Dlx2', 'Eomes', 'Tbr2'],
                'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfsr1'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2-Ab1', 'H2-Aa'], 
                'Undefined': ['Gadd45b', 'Ddit3', 'Htra2', 'Nr1d1'], 
                'EMT': ['Cdh2', 'Cdh11', 'Itga5', 'Itgb6', 'Sdc1', 'Fsp1', 'Vim', 'Ctnnb1', 'Fn1', 'Lama5', 'Col1a1', 'Snail1', 'Zeb1', 'Foxc2', 'Gsc']} 

In [ ]:
for k, v in markers1.items():
    genes_in_data = []
    for gene in v:
        if gene in adata.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
markers2 = {'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfs1r'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2-Ab1', 'H2-Aa']} 

In [ ]:
for k, v in markers2.items():
    genes_in_data = []
    for gene in v:
        if gene in adata.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, vmax=8, cmap='YlGnBu')

In [ ]:
markers3 = {'Macrophages': ['Ms4a7', 'Pf4', 'Dab2', 'C1qa', 'Mrc1', 'C1qc', 'Cfs1r'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes/Neutrophils': ['Cd44', 'Gsr', 'Thbs1', 'Ccl6', 'Msrb1'], 
                'B cells': ['Cd79a', 'Cd79b', 'Igkc', 'Ebf1'], 
                'T cells': ['Cd3d', 'Cd4', 'Trac', 'Lat'],
                'NK': ['Gzmb', 'Cd7', 'Ctsw', 'Xcl1', 'Nkg7'], 
                'DC': ['Plbd1', 'Naaa', 'Ppt1', 'H2.DMa', 'H2.DMb1', 'H2-DMa', 'H2-DMb1'], 
                'Epiplexus': ['Cd9', 'Syngr1', 'Sparc', 'Timp2', 'Olfml3', 'Plxdc2', 'Hexa', 'Gpr34', 
                              'Mpeg1', 'Cd81', 'Golm1', 'Adamts1', 'Serpine2', 'Cst7']} 

In [ ]:
for k, v in other_markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, vmax=8, size=20, cmap='YlGnBu')

In [ ]:
adata

# Removing doublets

In [ ]:
adata_edit = adata[(adata.obs['leiden_0.8']!='0')]

adata_noDBL = adata_all[adata_edit.obs.index]

In [ ]:
adata_noDBL.write_h5ad('noDBL_Immune_Postnatal_unfiltered.h5ad')

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_unfiltered.h5ad')

In [ ]:
adata_noDBL

## Ploting data before QC

In [ ]:
sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'],
             ncols=2, rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_noDBL_rawdata.svg')

In [ ]:
sc.pl.violin(adata_noDBL, ["Xist-counts", "percent_chrY"], jitter=0, groupby = 'batch', rotation= 90, palette=my_c)

In [ ]:
sc.tl.pca(adata_noDBL, svd_solver='arpack')
sc.pp.neighbors(adata_noDBL, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_noDBL)

In [ ]:
%matplotlib inline


sc.pl.umap(adata_noDBL, color=['Gender', 'Xist-counts'], vmax=1, palette=my_c)

In [ ]:
%matplotlib inline


sc.pl.umap(adata_noDBL, color=['Gender', 'percent_chrY'], vmax=0.1)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(18,12))
sns.countplot(y = 'batch',data = adata_noDBL.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata_noDBL.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
    
fig.savefig('barplots_immune_noDBL_rawdata.svg')

In [ ]:
sc.pl.highest_expr_genes(adata_noDBL,n_top=20)

In [ ]:
sc.pl.violin(adata_noDBL, ['S_score', 'G2M_score'], jitter=0, groupby = 'batch', rotation=90, palette=my_c)

In [ ]:
adata_noDBL

# QC

## Data normalization and logaritmization

In [ ]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata_noDBL,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata_noDBL)

**Creating raw data object**

In [ ]:
adata_noDBL.raw = adata_noDBL.copy()

In [ ]:
adata_noDBL

## Filtering cells and genes

In [ ]:
sc.pp.filter_cells(adata_noDBL,min_genes = 250)
sc.pp.filter_genes(adata_noDBL,min_cells = 3)

In [ ]:
%matplotlib inline

sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_noDBL_prefilter.svg')

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['total_counts'] < 15000,:] 

In [ ]:
%matplotlib inline

sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_immune_noDBL_afterfilter.svg')

In [ ]:
print("Remaining cells %d" % adata_noDBL.n_obs)

In [ ]:
adata_noDBL

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata_noDBL.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata_noDBL.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
fig.savefig('barplots_immune_noDBL_afterfilter.svg', format='svg')

## Saving filtered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Immune_Postnatal_noDBL_filtered.h5ad')

## Loading filtered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

# Dimensionality redutcion

## HVG

In [ ]:
#HVGs
sc.pp.highly_variable_genes(adata_noDBL, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat_v3', n_top_genes=4000)

In [ ]:
adata_noDBL

In [ ]:
sc.pl.highly_variable_genes(adata_noDBL, save='_immune_noDBL_HVG.svg')

In [ ]:
var_genes_all = adata_noDBL.var.highly_variable

In [ ]:
adata_noDBL = adata_noDBL[:, var_genes_all]

In [ ]:
adata_noDBL

In [ ]:
sc.pp.scale(adata_noDBL,max_value=10)

## PCA

In [ ]:
sc.tl.pca(adata_noDBL, svd_solver='arpack')

In [ ]:
sc.pl.pca(adata_noDBL, color='batch', components = ['1,2','3,4','5,6','7,8'], ncols=2, wspace=0.8, palette=my_c, 
          save='_immune_noDBL_pca.svg')

In [ ]:
sc.pl.pca_variance_ratio(adata_noDBL, log=True, n_pcs=50, save='_immune_noDBL_n_pcs.svg')

# Clustering

In [ ]:
sc.pp.neighbors(adata_noDBL, n_pcs=26, n_neighbors=15)
sc.tl.umap(adata_noDBL)

In [ ]:
sc.pl.umap(adata_noDBL, color=["batch", "Age"], size=15, wspace=0.8, ncols=2, palette=my_c, 
           save='_immune_noDBL_precorrection.svg')

## Batch correction

### Harmony

In [ ]:
from harmony import harmonize

In [ ]:
Z = harmonize(adata_noDBL.obsm['X_pca'], adata_noDBL.obs, batch_key = 'batch', max_iter_harmony=20)
adata_noDBL.obsm['X_harmony'] = Z

In [ ]:
print("Calculating UMAP")
sc.pp.neighbors(adata_noDBL, n_pcs=26, n_neighbors=15, use_rep="X_harmony")
sc.tl.umap(adata_noDBL)

In [ ]:
print("Ploting UMAP")
sc.pl.umap(adata_noDBL, color=["batch", 'Age'], size=15, wspace=0.8, ncols=2, palette=my_c, 
           save='_immune_noDBL_aftercorrection.svg')

## Clustering

In [ ]:
# clustering
for res in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]:
    sc.tl.leiden(adata_noDBL, resolution=res,key_added = 'leiden_%s'%res)

In [ ]:
#plot
sc.pl.umap(adata_noDBL,color=['leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 
                        'leiden_0.8', 'leiden_1.0', 'leiden_1.2', 'leiden_1.4', 'leiden_1.6', 'leiden_1.8', 
                        'leiden_2.0'], ncols=4, wspace=1, cmap='my_c')

In [ ]:
%matplotlib inline

sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='leiden_0.1', save='_immune_noDBL_afterclustering.svg')

## Saving clsutered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading filtered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata_noDBL

### Leiden_0.1

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.1', method='wilcoxon', key_added = "wilcoxon_01", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_01" , groupby="leiden_0.1", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.2

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.2', method='wilcoxon', key_added = "wilcoxon_02", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_02" , groupby="leiden_0.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.3

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.3', method='wilcoxon', key_added = "wilcoxon_03", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_03" , groupby="leiden_0.3", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.4

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.4', method='wilcoxon', key_added = "wilcoxon_04", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_04" , groupby="leiden_0.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.5

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.5', method='wilcoxon', key_added = "wilcoxon_05", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_05" , groupby="leiden_0.5", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.6

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_06", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_06" , groupby="leiden_0.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.8

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.8', method='wilcoxon', key_added = "wilcoxon_08", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_08" , groupby="leiden_0.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.0

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.0', method='wilcoxon', key_added = "wilcoxon_1", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_1" , groupby="leiden_1.0", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.2

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.2', method='wilcoxon', key_added = "wilcoxon_12", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_12" , groupby="leiden_1.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.4

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.4', method='wilcoxon', key_added = "wilcoxon_14", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_14" , groupby="leiden_1.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.6

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.6', method='wilcoxon', key_added = "wilcoxon_16", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_16" , groupby="leiden_1.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.8

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.8', method='wilcoxon', key_added = "wilcoxon_18", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_18" , groupby="leiden_1.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_2.0

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_2.0', method='wilcoxon', key_added = "wilcoxon_2", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_2" , groupby="leiden_2.0", cmap="YlGnBu", vmin=0, vmax=10)

## Saving clsutered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading clustered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL_all = sc.read_h5ad('noDBL_Immune_Postnatal_unfiltered.h5ad')

In [ ]:
markers = {"Epithelial cells": ['Gata3','Atp10b','Cidea', "Krt18","Htr2c","Car12","Clic6", "Folr1","Igfbp2",'Otx2','Chchd10','Clu','Calml4'],
           "Mesenchymal cells": ['Pdgfrb','Mgp','Cfh','Dcn','Tgfbi','Moxd1','Cp','Bgn','Cxcl12','Col3a1'],
           "Endothelial cells": ["Egfl7","Kdr","Pecam1", "Cd34", "Plvap",'Cdh5','Flt1','Igfbp3','Ly6a','Ly6c1'],
           "Immune cells": ["C1qa",'Tyrobp','Fcer1g','Ctss','C1qb','Ptprc','C1qc','Cd83','Lyz2','Laptm5'],
           "NeuroGlia": ['Tubb2b','Gpm6b','Rtn1','Olig1','Fabp7','Bcan','Cnp','Scrg1','Opcml','Pcsk1n'], 
           "NIPC": ["Cdca7", "2810417H13Rik", "Rrm2", "Ube2c", "Top2a", "Birc5", "P2rx3"]}

In [ ]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=6)

In [ ]:
markers1 = {'Ciliogenesis': ['Shisa8', 'Ccdc67', 'Ccno', 'Mcidas'], 
                'Mesenchymal cells': ['Dcn', 'Adam12', 'Cxcl12', 'Lum'], 
                'Epithelial cells': ['Htr2c', 'Enpp2', 'Kcnj13', 'Aqp1', 'Gata3'], 
                'Endothelial cells': ['Plvap', 'Flt1', 'Pecam1'],
                'Neuro / Glial': ['Slc1a3', 'Olig1', 'Olig2', 'Dlx1', 'Dlx2', 'Eomes', 'Tbr2'],
                'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfsr1'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2-Ab1', 'H2-Aa']} 

In [ ]:
for k, v in markers1.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
markers2 = {'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfs1r'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2-Ab1', 'H2-Aa']} 

In [ ]:
for k, v in markers2.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, vmax=8, cmap='YlGnBu')

In [ ]:
gene1 = 'Cd3g'
gene2 = 'Txk'
gene3 = 'Cxcr6'
gene4 = 'Cd79a'
gene5 = 'Tcf7'
gene6 = 'Ebf1'
gene7 = 'Trac'
gene8 = 'Cd7'
gene9 = 'Xcl1'

adata_noDBL.obs['CoEx'] = ((adata_noDBL.raw[:,'{}'.format(gene1)].X.todense() > 0) |
                    (adata_noDBL.raw[:,'{}'.format(gene2)].X.todense() > 0) |
                    (adata_noDBL.raw[:,'{}'.format(gene3)].X.todense() > 0) | 
                    (adata_noDBL.raw[:,'{}'.format(gene4)].X.todense() > 0) | 
                    (adata_noDBL.raw[:,'{}'.format(gene5)].X.todense() > 0) | 
                    (adata_noDBL.raw[:,'{}'.format(gene6)].X.todense() > 0) |
                    (adata_noDBL.raw[:,'{}'.format(gene7)].X.todense() > 0) |
                    (adata_noDBL.raw[:,'{}'.format(gene8)].X.todense() > 0) |
                    (adata_noDBL.raw[:,'{}'.format(gene9)].X.todense() > 0))

In [ ]:
cells = []

for cell in adata_noDBL.obs_names:
    if adata_noDBL.obs['CoEx'].loc[cell] == True:
        
        cells.append(cell)

In [ ]:
adata_noDBL.obs['CoEx'] = adata_noDBL.obs['CoEx'].astype('category')

In [ ]:
sc.pl.umap(adata_noDBL, color=['CoEx', 'Txk', 'Cd3d', 'Cxcr6', 'Cd79a', 'Cd19', 'Vpreb3', 'Il7r', 'Cd7', 'Ctsw'], 
           palette=['white', 'Black'], cmap='YlGnBu')

In [ ]:
i = 0

for v in adata_noDBL.obs['CoEx']:
    if v == True:
        i += 1
    else:
        continue

i

In [ ]:
other_markers = {'Macrophages': ['Ms4a7', 'Pf4', 'Dab2', 'C1qa', 'Mrc1', 'C1qc', 'Cfs1r'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes/Neutrophils': ['Ace', 'Ly6c2', 'Cd44', 'Gsr', 'Thbs1', 'Ccl6', 'Msrb1'], 
                'B cells': ['Cd79a', 'Cd79b', 'Igkc', 'Ebf1'], 
                'T cells': ['Cd3d', 'Cd4', 'Trac', 'Lat'],
                'NK': ['Gzmb', 'Cd7', 'Ctsw', 'Xcl1', 'Nkg7'], 
                'DC': ['Plbd1', 'Naaa', 'Ppt1', 'H2.DMa', 'H2.DMb1', 'H2-DMa', 'H2-DMb1'], 
                'Epiplexus': ['Cd9', 'Syngr1', 'Sparc', 'Timp2', 'Olfml3', 'Plxdc2', 'Hexa', 'Gpr34', 
                              'Mpeg1', 'Cd81', 'Golm1', 'Adamts1', 'Serpine2', 'Cst7'], 
                'Plasmocytes': ['Clec4c', 'Nrp1', 'Il7r', 'Itgax', 'Ptprc', 'Ly6c', 'Bst2', 'Siglech']} 

In [ ]:
for k, v in other_markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, vmax=8, size=20, cmap='YlGnBu')

In [ ]:
Monocytes = {'Classical monocytes': ['Adre1', 'Csf1r', 'Ly6c2', 'Ccr2', 'Chil3'],
             'Non-classical monocytes': ['Ace', 'Eno3']}

In [ ]:
for k, v in Monocytes.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, vmax=8, size=20, cmap='YlGnBu')

In [ ]:
DC = {'cDC2': ['Cd209a', 'Tnfsf9', 'Tnip3', 'Kcne3', 'Flt3'],
      'cDC1': ['Itgae', 'Xcr1', 'Sept3', 'Gcsam', 'Cleac9a', 'Pianp', 'Ffar4', 'Plpp1', 'Cxx1a', 'Cxx1b', 'Tlr3', 'Snx22'],
      'pDC': ['Ccr9', 'Cd300c', 'Cox6a2', 'Runx2', 'Klk1', 'Atp2a1', 'Pacsin1', 'Sh3bgr', 'Ly6d', 'Lefty1', 'Paqr5'],
      'migDC': ['Il4i1', 'Ccr7', 'Fscn1', 'Nudt17', 'Socs2', 'Cacnb3', 'Ccl22', 'Mreg', 'Tnfrsf4', 'Apol7c', 'Il12b', 'Slco5a1']}

In [ ]:
for k, v in DC.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, vmax=2, size=20, cmap='YlGnBu')

In [ ]:
adata_noDBL

# Cluster annotation

In [ ]:
adata_noDBL.obs["Cell_type"] = adata_noDBL.obs["leiden_1.2"].replace({"16": "Proliferating macrophages",
                                                                      "6": "Macrophages", 
                                                                      "8": "Macrophages",
                                                                      "1": "Macrophages",
                                                                      "11": "Macrophages",
                                                                      "0": "Macrophages",
                                                                      "7": "Macrophages", 
                                                                      "4": "cDCs",
                                                                      "9": "cDCs",
                                                                      "22": "Neutrophils",
                                                                      "17": "Classical monocytes",
                                                                      "21": "Non classical monocytes",
                                                                      "24": "B cells",
                                                                      "20": "NK",
                                                                      "23": "T cells",
                                                                      "10": "Macrophages",
                                                                      "3": "Macrophages",
                                                                      "5": "Macrophages",
                                                                      "14": "Macrophages",
                                                                      "19": "Macrophages",
                                                                      "2": "Macrophages",
                                                                      "12": "Macrophages",
                                                                      "15": "Epiplexus cells",
                                                                      "13": "Macrophages",
                                                                      "18": "Macrophages"}) 


In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'Cell_type', method='wilcoxon', key_added = "wilcoxon_Cell_type", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=10, save='_immune_noDBL_15_nofiltered.svg')

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Age', 'batch'], size=15, ncols=4, wspace=0.7, 
           save='_immune_celltypes_age_batch.svg', palette=my_c)

In [ ]:
sc.tl.filter_rank_genes_groups(adata_noDBL, groupby='Cell_type', key = "wilcoxon_Cell_type", 
                               key_added='wilcoxon_filtered_Cell_type',
                               min_in_group_fraction=0.1,max_out_group_fraction=0.5,min_fold_change=0.25)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_filtered_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_immune_filtered_15.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=10, key="wilcoxon_filtered_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_immune_filtered_10.svg')

In [ ]:
adata_noDBL.obs['P2/P3'] = False
adata_noDBL.obs['P11'] = False
adata_noDBL.obs['P60'] = False

In [ ]:
ages = ['P2/P3', 'P11', 'P60']

In [ ]:
for p in ages:
    for cell in adata_noDBL.obs_names:
        if adata_noDBL.obs['Age'].loc[cell] == p:
            adata_noDBL.obs[p].loc[cell] = p
        else:
            adata_noDBL.obs[p].loc[cell] = np.nan

In [ ]:
sc.pl.umap(adata_noDBL, color=ages, size=15, wspace=0.5, cmap='YlGnBu_r', na_color="#FF000000", 
           save='_immune_ages.svg')

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, groupby="Cell_type", method='wilcoxon', n_genes=50)

clust_df = pd.DataFrame(adata_noDBL.uns['rank_genes_groups']['names'])
clust_df.to_csv("Immune_Top_50_Postnatal_mouse_marker_genes_CP.csv")

### Plots cell markers

In [ ]:
markers = ['Aif1', 'Adgre1', 'Fcgr1', 'Mertk', 
           'Sparc', 'Syngr1', 'Crybb1', 'Serpine2',
           'Top2a', 'Birc5', 'Pclaf', 'Mki67',
           'H2-DMb1', 'Cd72', 'Tnfsf9', 'Tnip3',
           'S100a9', 'S100a8', 'Cstdc4', 'Stfa2',
           'Emilin2', 'Ccr2', 'Ly6c2', 'Chil3',
           'Adgre4', 'Gfpt1', 'Eno3', 'Ace',
           'Ebf1', 'Cd79a', 'Ly6d', 'Igkc',
           'Gzmb', 'Cd7', 'Klre1', 'Klrb1c',
           'Cd3d', 'Lat', 'Trac', 'Cers4']

In [ ]:
sns.set_style({'axes.facecolor': 'white',
 'axes.edgecolor': '.8',
 'axes.grid': False,
 'axes.axisbelow': 'line',
 'axes.labelcolor': 'black',
 'figure.facecolor': 'white',
 'grid.color': '.8',
 'grid.linestyle': '-',
 'text.color': 'black',
 'xtick.color': 'black',
 'ytick.color': 'black',
 'xtick.direction': 'out',
 'ytick.direction': 'out',
 'patch.edgecolor': 'black',
 'patch.force_edgecolor': False,
 'image.cmap': 'viridis',
 'font.family': ['sans-serif'],
 'font.sans-serif': ['DejaVu Sans',
  'Bitstream Vera Sans',
  'Computer Modern Sans Serif',
  'Lucida Grande',
  'Verdana',
  'Geneva',
  'Lucid',
  'Arial',
  'Helvetica',
  'Avant Garde',
  'sans-serif'],
 'xtick.bottom': True,
 'xtick.top': False,
 'ytick.left': True,
 'ytick.right': False,
 'axes.spines.left': True,
 'axes.spines.bottom': True,
 'axes.spines.right': True,
 'axes.spines.top': True}
)

vp = sc.pl.rank_genes_groups_stacked_violin(adata_noDBL, var_names=markers, groupby='Cell_type', key='wilcoxon_Cell_type', categories_order=cells,
                                            row_palette=color_order, dendrogram=False,  yticklabels = True, return_fig=True, figsize=(13,5))

vp.style(ylim=(0,6)).show()

vp.savefig('stacked_violin_NEW_noDBL_celltype_markers_immune_cells.svg', format='svg', dpi=180)
axes_dict = vp.get_axes()
print(axes_dict)

In [ ]:
cells = ['Macrophages', 'Epiplexus cells', 'Proliferating macrophages', 'cDCs', 'Neutrophils', 'Classical monocytes', 'Non classical monocytes', 'B cells', 
         'NK', 'T cells']

In [ ]:
color_order = ['#fe864a', '#95d0fc', '#fbdd7e', '#c9d179', '#ffbacd', '#388004', '#76cd26', '#cfaf7b', '#c760ff', '#d6b4fc']

In [ ]:
adata_noDBL.uns['Cell_type_colors']

In [ ]:
sc.pl.violin(adata_noDBL, ['total_counts','n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="Cell_type", rotation=90, stripplot=False, palette=my_c, save='_immune_celltypes_metrics.svg')

In [ ]:
adata_noDBL.obs['Log10_nGenes'] = np.log10(adata_noDBL.obs['n_genes_by_counts'])

In [ ]:
vars_plot = ['total_counts', 'Log10_nGenes', 'n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb']

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 6), sharex=True)

groupby_var = "Cell_type"

# Loop over genes and groupby variables to create violin plots
for i, v in enumerate(vars_plot):
    if i != 1:
        ax = axes[i] if 6 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata_noDBL, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0)  # Adjust the y-limits as needed for each gene
    else:
        ax = axes[i] if 5 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata_noDBL, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0, top=5)  # Adjust the y-limits as needed for each gene

# Add a common x-axis label and display the plot
plt.tight_layout()  # Adjust subplot layout to prevent overlap
plt.show()

fig.savefig('violinplots_immune_metrics.svg', format='svg', dpi=180)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 5), sharex=True)

groupby_var = "batch"

# Loop over genes and groupby variables to create violin plots
for i, v in enumerate(vars_plot):
    if i != 1:
        ax = axes[i] if 6 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata_noDBL, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0)  # Adjust the y-limits as needed for each gene
    else:
        ax = axes[i] if 5 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata_noDBL, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0, top=5)  # Adjust the y-limits as needed for each gene

# Add a common x-axis label and display the plot
plt.tight_layout()  # Adjust subplot layout to prevent overlap
plt.show()

fig.savefig('violinplots_immune_samples_metrics.svg', format='svg', dpi=180)

In [ ]:
def batch_pct_cellType(adata_noDBL_file,groupby_key,legend_key,pos):
    tmp = pd.crosstab(adata_noDBL_file.obs[groupby_key],adata_noDBL_file.obs[legend_key], normalize='index')   
    tmp.plot.bar(stacked=True,rot= 90, grid=False, color=my_c,
                 xlabel='',ax=pos).legend(bbox_to_anchor=[1.15,1])

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, constrained_layout=True,figsize=(8,10))
batch_pct_cellType(adata_noDBL,'Cell_type','Age',ax1)
batch_pct_cellType(adata_noDBL,'Cell_type','batch',ax2)

fig.savefig('barplots_immune_celltypes_age.svg', format='svg')

### Cell proportions

In [ ]:
ct = pd.crosstab(adata_noDBL.obs['batch'],adata_noDBL.obs['Cell_type'], normalize='index')

In [ ]:
ct['Age'] = None

In [ ]:
for b in ct.index:
    if 'P2' in b:
        ct['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ct['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ct['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ct['Age'].loc[b] = 'P60'

In [ ]:
ctt = ct.reset_index().melt(id_vars=['batch'])[:-8]

ctt.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

ctt['Age'] = None

ctt = ctt.set_index(['batch'])

ctt.index

In [ ]:
for b in ctt.index:
    if 'P2' in b:
        ctt['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ctt['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ctt['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ctt['Age'].loc[b] = 'P60'

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=True, ax=ax1)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.38, ax=ax2)


ax1.set_ylim(0.6,1.0)   # those limits are fake
ax2.set_ylim(0,0.21)

sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates

kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])

ax1.set_yticks(np.arange(0.6,1.01, 0.10))
ax2.set_yticks(np.arange(0,0.21, 0.10))

from textwrap import wrap
names = [cell for cell in ctt['Cell_type'].unique()]

names = [ '\n'.join(wrap(l, 16)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[3]), (handles[1], handles[4]), (handles[2], handles[5])],
          labels=['P2/P3', 'P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=16)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

fig.savefig('boxplots_immune_all_celltypes.svg')

### Statistics on cell proportions

In [ ]:
ct_mann_whitney_u = ct.copy()

In [ ]:
ct_mann_whitney_u['Age'] = None

In [ ]:
for b in ct_mann_whitney_u.index:
    if 'P2' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P60'

In [ ]:
ctt_mann_whitney_u = ct_mann_whitney_u.reset_index().melt(id_vars=['batch'])[:-8]

In [ ]:
ctt_mann_whitney_u.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

In [ ]:
ctt_mann_whitney_u['Age'] = None

In [ ]:
ctt_mann_whitney_u = ctt_mann_whitney_u.set_index(['batch'])

In [ ]:
for b in ctt_mann_whitney_u.index:
    if 'P2' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P60'

In [ ]:
df_undone = {}

for ctype in ctt_mann_whitney_u['Cell_type'].unique():
    for age in ctt_mann_whitney_u['Age'].unique():
        print(ctype, age)
        sdf = ctt_mann_whitney_u[(ctt_mann_whitney_u['Cell_type'] == ctype) & (ctt_mann_whitney_u['Age'] == age)]
        values = list(sdf['Percentage of cells'])
        df_undone[f"{ctype}_{age}"] = values


In [ ]:
from scipy.stats import mannwhitneyu

mann_whitney_u_res = {}

for ctype in ctt_mann_whitney_u['Cell_type'].unique():
    vals_list = []
    for k,v in df_undone.items():
        if ctype in k:
            print(k)
            vals_list.append(df_undone[k])
    f_statistic, p_value = mannwhitneyu(vals_list[0], vals_list[1], method='exact')
    mann_whitney_u_res[ctype] = [f_statistic, p_value]

In [ ]:
mann_whitney_u_res_df = pd.DataFrame(mann_whitney_u_res.items(), columns=['Cell_type', 'Values']).set_index('Cell_type')

In [ ]:
mann_whitney_u_res_df['Values'].iloc[[0]]

In [ ]:
mann_whitney_u_res_df['f_statistics'] = None
mann_whitney_u_res_df['p_value'] = None

In [ ]:
for i in range(0, len(mann_whitney_u_res_df['Values'])):
    mann_whitney_u_res_df['f_statistics'].iloc[[i]] = mann_whitney_u_res_df['Values'][i][0]
    mann_whitney_u_res_df['p_value'].iloc[[i]] = mann_whitney_u_res_df['Values'][i][1]

In [ ]:
mann_whitney_u_res_df.drop(columns='Values', inplace=True)
mann_whitney_u_res_df

In [ ]:
mann_whitney_u_res_df.to_csv('Mann_Whitney_u_cell_proportions_statistics_immune.csv')

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt_mann_whitney_u['Cell_type'], y = ctt_mann_whitney_u['Percentage of cells'], data = ctt_mann_whitney_u, hue=ctt_mann_whitney_u['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt_mann_whitney_u, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.35, ax=ax1)

sns.boxplot(x = ctt_mann_whitney_u['Cell_type'], y = ctt_mann_whitney_u['Percentage of cells'], data = ctt_mann_whitney_u, hue=ctt_mann_whitney_u['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt_mann_whitney_u, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.3551, ax=ax2)


ax1.set_ylim(0.6,1)   # those limits are fake
ax2.set_ylim(0,0.2)
sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates

kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])
ax1.set_yticks(np.arange(0.60,1, 0.1))
ax2.set_yticks(np.arange(0,0.21, 0.05))

from textwrap import wrap
names = [cell for cell in ctt_mann_whitney_u['Cell_type'].unique()]

names = [ '\n'.join(wrap(l, 16)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[2]), (handles[1], handles[3])],
          labels=['P2/P3/P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=16)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

fig.savefig('mann_whitney_u_boxplots_noDBL_all_celltypesgroups_immune.svg')

## Saving annotated data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

## Loading annotated data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Age', 'batch'], size=15, ncols=4, wspace=0.8, 
           save='_immune_celltypes_age_batch.svg')

# Gender inference based on Xist-counts and percent_chrY

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL.obs['Xist-counts']

In [ ]:
adata_noDBL.obs['percent_chrY']

In [ ]:
adata_noDBL.obs['Female'] = (adata_noDBL.obs['Xist-counts'] > 0)

In [ ]:
adata_noDBL.obs['Female']

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Female'], ncols=4, size=20, palette= my_c,wspace=0.7, save='_Celltypes_gender_immune.svg')

In [ ]:
sc.pl.umap(adata_noDBL, color=['percent_chrY', 'Xist-counts'], ncols=4, size=20, palette= my_c,wspace=0.7, vmax=0.2, save='_Percent_chrY_Xits_counts_immune.svg')